In [1]:
import torch
import numpy as np
import pandas as pd
from haversine import haversine, Unit
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder, StandardScaler


/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 100

# 1. Load Dataset

In [3]:
trainpath = f'../../../data/top30groups/LongLatCombined/train1/train{partition}.csv'
testpath = f'../../../data/top30groups/LongLatCombined/test1/test{partition}.csv'
traindata = pd.read_csv(trainpath, encoding='ISO-8859-1')
testdata = pd.read_csv(testpath, encoding='ISO-8859-1')

In [4]:
combined = pd.concat([traindata, testdata], axis = 0)

### Find unique locations and construct global graph

In [5]:
# Extract unique locations for node creation
combined['location'] = list(zip(combined['longitude'], combined['latitude']))
unique_locations = combined['location'].drop_duplicates().reset_index(drop=True)

# Map locations to an identity
location2id = {loc: idx for idx, loc in enumerate(unique_locations)}
combined['location_id'] = combined['location'].map(location2id)

# Encode labels
le = LabelEncoder()
combined['label'] = le.fit_transform(combined['gname'])

# Get global node features
coords = np.array([list(loc) for loc in unique_locations])  # [1790, 2]
print("Feature Matrix shape: ", coords.shape)

# Standardize features
scaler = StandardScaler()
x_global = scaler.fit_transform(coords)  # standardized features

# Build global edge list using 1km Haversine
edges = []
coords_latlon = [(lat, lon) for lon, lat in unique_locations]
for i in range(len(coords_latlon)):
    for j in range(i + 1, len(coords_latlon)):
        if haversine(coords_latlon[i], coords_latlon[j], Unit.KILOMETERS) <= 1.0:
            edges.append((i, j))
            edges.append((j, i))

global_edge_index = torch.tensor(edges, dtype=torch.long).T  # shape [2, num_edges]


Feature Matrix shape:  (1790, 2)


In [6]:
global_edge_index.shape

torch.Size([2, 242])

In [7]:
unique_nodes = torch.unique(global_edge_index)
print("Nodes with at least one neighbor: ", len(unique_nodes))

Nodes with at least one neighbor:  161


### Creating subgraphs for each node depending on its neighbors

In [8]:
def get_subgraph(center_id, edge_index, x_global):
    # Get neighbors (indices) of center node
    neighbors = edge_index[1][edge_index[0] == center_id]
    node_ids = torch.cat([torch.tensor([center_id]), neighbors]).unique()

    # Remap node indices locally
    id_map = {old_id.item(): i for i, old_id in enumerate(node_ids)}
    new_edges = []
    for source, destination in zip(*edge_index):
        if source in node_ids and destination in node_ids:
            new_edges.append((id_map[source.item()], id_map[destination.item()]))

    # If no edges exist, add a self-loop
    if len(new_edges) == 0:
        center_local_idx = 0  # only node in subgraph
        new_edges = [(0, 0)]
    else:
        center_local_idx = id_map[center_id.item()]

    sub_x = x_global[node_ids]
    sub_edge_index = torch.tensor(new_edges).T

    return sub_x, sub_edge_index, center_local_idx


### Subgraphs for train

In [9]:
from torch_geometric.data import Data

traindata_list = []
for _, row in traindata.iterrows():
    center_id = location2id[(row['longitude'], row['latitude'])]
    label = le.transform([row['gname']])[0]
    
    x, edge_index, center_idx = get_subgraph(torch.tensor(center_id), global_edge_index, torch.tensor(x_global, dtype=torch.float))
    
    traindata_obj = Data(x=x, edge_index=edge_index, y=torch.tensor(label), center=center_idx)
    traindata_list.append(traindata_obj)


### Subgraphs for test

In [10]:
test_data_list = []
for _, row in testdata.iterrows():
    loc = (row['longitude'], row['latitude'])
    
    # Skip if location not in mapping (just in case)
    if loc not in location2id:
        continue
    
    center_id = location2id[loc]
    label = le.transform([row['gname']])[0]
    
    x, edge_index, center_idx = get_subgraph(
        torch.tensor(center_id),
        global_edge_index,
        torch.tensor(x_global, dtype=torch.float)
    )

    testdata_obj = Data(x=x, edge_index=edge_index, y=torch.tensor(label), center=center_idx)
    test_data_list.append(testdata_obj)

### GCN Model

In [11]:
import torch.nn as nn


class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, activation_fn=F.relu):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.classifier = nn.Linear(hidden_channels, out_channels)
        self.activation_fn = activation_fn

    def forward(self, batch):
        x, edge_index = batch.x, batch.edge_index
        #x = self.conv1(x, edge_index)
        #x = F.relu(x)
        x = self.activation_fn(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)

        # batch.ptr[:-1] extracts central node index
        center_embeddings = x[batch.ptr[:-1]]
        out = self.classifier(center_embeddings)
        return F.log_softmax(out, dim=1)

### Training and Testing

In [12]:
from sklearn.model_selection import train_test_split

train_set, val_set = train_test_split(traindata_list, test_size=0.2, random_state=42)


### Define batches

In [13]:
def train(loader):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    for batch in loader:
        batch = batch.to(device)
        out = model(batch)
        pred = out.argmax(dim=1)
        correct += (pred == batch.y).sum().item()
        total += batch.y.size(0)
    return correct / total

In [14]:
from itertools import product

best_model_state = None
patience = 500

import torch.nn.functional as F

activation_map = {
    'relu': F.relu,
    'tanh': F.tanh
}

param_dist = {
    'h1': [10, 50, 100, 150, 200, 300],
    'activations': ['relu', 'tanh'],
    'lrs': [0.0001, 0.001, 0.01],
    #'alphas': [1e-5, 1e-4, 1e-3, 1e-2]
    'batch_sizes': [128, 256, 512]
    }


keys, values = zip(*param_dist.items())
combinations = [dict(zip(keys, v)) for v in product(*values)]

import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

best_params = None
num_classes = len(le.classes_)

global_best_val_acc = 0.0
global_best_model_state = None
global_best_params = None
global_best_epoch = 0

for i, params in enumerate(combinations):
    print(f"Combination {i+1}/{len(combinations)}: {params}")

    act_fn = activation_map[params["activations"]]

    model = GCN(in_channels=2, hidden_channels=params["h1"], out_channels=len(le.classes_), activation_fn = act_fn).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lrs"], weight_decay=5e-4)
    criterion = torch.nn.CrossEntropyLoss()

    train_loader = DataLoader(train_set, batch_size=params["batch_sizes"], shuffle=True)
    val_loader = DataLoader(val_set, batch_size=params["batch_sizes"])
    test_loader = DataLoader(test_data_list, batch_size=params["batch_sizes"])

    patience_counter = 0
    best_val_acc = 0
    best_epoch = 0

    for epoch in range(1, 1500):
        avg_loss = train(train_loader)

        train_acc = evaluate(train_loader)
        val_acc = evaluate(val_loader)

        if epoch % 50 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            best_model_state = model.state_dict()
            best_params = params
            best_epoch = epoch

            if val_acc > global_best_val_acc:
                global_best_val_acc = val_acc
                global_best_model_state = model.state_dict()
                global_best_params = params
                global_best_epoch = epoch
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    print("Best epoch: ", best_epoch)
    
final_model = GCN(
    in_channels=2,
    hidden_channels=global_best_params['h1'],
    out_channels=num_classes,
    activation_fn=activation_map[global_best_params['activations']]
).to(device)

final_model.load_state_dict(global_best_model_state)

test_loader = DataLoader(test_data_list, batch_size=global_best_params["batch_sizes"])
test_acc = evaluate(test_loader)

print("\nBest hyperparameters:", global_best_params)
print(f"Best epoch: {global_best_epoch}")
print(f"Final Test Accuracy after early stopping: {test_acc:.4f}")

Combination 1/108: {'h1': 10, 'activations': 'relu', 'lrs': 0.0001, 'batch_sizes': 128}
Epoch 001 | Loss: 3.4414 | Train Acc: 0.0000 | Val Acc: 0.0000
Epoch 050 | Loss: 3.1766 | Train Acc: 0.1679 | Val Acc: 0.1619
Epoch 100 | Loss: 2.8410 | Train Acc: 0.2137 | Val Acc: 0.1714
Epoch 150 | Loss: 2.4471 | Train Acc: 0.2595 | Val Acc: 0.2048
Epoch 200 | Loss: 2.1755 | Train Acc: 0.3119 | Val Acc: 0.2548
Epoch 250 | Loss: 1.9897 | Train Acc: 0.3637 | Val Acc: 0.2929
Epoch 300 | Loss: 1.8339 | Train Acc: 0.4173 | Val Acc: 0.3286
Epoch 350 | Loss: 1.7233 | Train Acc: 0.4565 | Val Acc: 0.3833
Epoch 400 | Loss: 1.6424 | Train Acc: 0.4673 | Val Acc: 0.3881
Epoch 450 | Loss: 1.5383 | Train Acc: 0.4964 | Val Acc: 0.4071
Epoch 500 | Loss: 1.4651 | Train Acc: 0.5232 | Val Acc: 0.4548
Epoch 550 | Loss: 1.3925 | Train Acc: 0.5381 | Val Acc: 0.4643
Epoch 600 | Loss: 1.3396 | Train Acc: 0.5542 | Val Acc: 0.4929
Epoch 650 | Loss: 1.2879 | Train Acc: 0.6173 | Val Acc: 0.5595
Epoch 700 | Loss: 1.2359 | Tra

In [ ]:
"""
Combination 98/108: {'h1': 300, 'activations': 'relu', 'lrs': 0.01, 'batch_sizes': 256}
Epoch 001 | Loss: 2.4808 | Train Acc: 0.3810 | Val Acc: 0.3571
Epoch 050 | Loss: 0.5538 | Train Acc: 0.7738 | Val Acc: 0.8262
Epoch 100 | Loss: 0.4902 | Train Acc: 0.8012 | Val Acc: 0.8452
Epoch 150 | Loss: 0.4965 | Train Acc: 0.8060 | Val Acc: 0.8214
Epoch 200 | Loss: 0.4310 | Train Acc: 0.8000 | Val Acc: 0.8357
Epoch 250 | Loss: 0.4086 | Train Acc: 0.8381 | Val Acc: 0.8452
Epoch 300 | Loss: 0.3934 | Train Acc: 0.8298 | Val Acc: 0.8476
Epoch 350 | Loss: 0.3780 | Train Acc: 0.8440 | Val Acc: 0.8429
Epoch 400 | Loss: 0.3619 | Train Acc: 0.8393 | Val Acc: 0.8452
Epoch 450 | Loss: 0.3982 | Train Acc: 0.8560 | Val Acc: 0.8595
Epoch 500 | Loss: 0.4124 | Train Acc: 0.8315 | Val Acc: 0.8524
Epoch 550 | Loss: 0.3548 | Train Acc: 0.8548 | Val Acc: 0.8500
Epoch 600 | Loss: 0.3325 | Train Acc: 0.8435 | Val Acc: 0.8643
Epoch 650 | Loss: 0.7309 | Train Acc: 0.7768 | Val Acc: 0.8190
Epoch 700 | Loss: 0.3780 | Train Acc: 0.8577 | Val Acc: 0.8714
Epoch 750 | Loss: 0.3405 | Train Acc: 0.8536 | Val Acc: 0.8667
Epoch 800 | Loss: 0.3307 | Train Acc: 0.8685 | Val Acc: 0.8667
Epoch 850 | Loss: 0.3512 | Train Acc: 0.8643 | Val Acc: 0.8857
Epoch 900 | Loss: 0.3435 | Train Acc: 0.8637 | Val Acc: 0.8548
Epoch 950 | Loss: 0.3884 | Train Acc: 0.8482 | Val Acc: 0.8595
Epoch 1000 | Loss: 0.3356 | Train Acc: 0.8452 | Val Acc: 0.8524
Epoch 1050 | Loss: 0.3427 | Train Acc: 0.8554 | Val Acc: 0.8690
Epoch 1100 | Loss: 0.3342 | Train Acc: 0.8619 | Val Acc: 0.8952
Epoch 1150 | Loss: 0.3189 | Train Acc: 0.8673 | Val Acc: 0.8714
Epoch 1200 | Loss: 0.3272 | Train Acc: 0.8601 | Val Acc: 0.8738
Epoch 1250 | Loss: 0.3235 | Train Acc: 0.8435 | Val Acc: 0.8571
Epoch 1300 | Loss: 0.3198 | Train Acc: 0.8411 | Val Acc: 0.8571
Epoch 1350 | Loss: 0.3334 | Train Acc: 0.8488 | Val Acc: 0.8452
Epoch 1400 | Loss: 0.3105 | Train Acc: 0.8655 | Val Acc: 0.8571
Epoch 1450 | Loss: 0.2984 | Train Acc: 0.8560 | Val Acc: 0.8786
Best epoch:  1100
"""

In [15]:
#Best hyperparameters: {'h1': 150, 'activations': 'tanh', 'lrs': 0.001, 'batch_sizes': 128}
#Final Test Accuracy after early stopping: 0.7978

In [16]:
"""Combination 31/108: {'h1': 50, 'activations': 'tanh', 'lrs': 0.001, 'batch_sizes': 128}
Combination 32/108: {'h1': 50, 'activations': 'tanh', 'lrs': 0.001, 'batch_sizes': 256}
Combination 35/108: {'h1': 50, 'activations': 'tanh', 'lrs': 0.01, 'batch_sizes': 256}
Combination 36/108: {'h1': 50, 'activations': 'tanh', 'lrs': 0.01, 'batch_sizes': 512}
Combination 49/108: {'h1': 100, 'activations': 'tanh', 'lrs': 0.001, 'batch_sizes': 128}
Combination 50/108: {'h1': 100, 'activations': 'tanh', 'lrs': 0.001, 'batch_sizes': 256}
Combination 51/108: {'h1': 100, 'activations': 'tanh', 'lrs': 0.001, 'batch_sizes': 512}
Combination 54/108: {'h1': 100, 'activations': 'tanh', 'lrs': 0.01, 'batch_sizes': 512}
Combination 58/108: {'h1': 150, 'activations': 'relu', 'lrs': 0.001, 'batch_sizes': 128}
Combination 59/108: {'h1': 150, 'activations': 'relu', 'lrs': 0.001, 'batch_sizes': 256}
Combination 60/108: {'h1': 150, 'activations': 'relu', 'lrs': 0.001, 'batch_sizes': 512}
Combination 67/108: {'h1': 150, 'activations': 'tanh', 'lrs': 0.001, 'batch_sizes': 128}
Combination 68/108: {'h1': 150, 'activations': 'tanh', 'lrs': 0.001, 'batch_sizes': 256}
Combination 69/108: {'h1': 150, 'activations': 'tanh', 'lrs': 0.001, 'batch_sizes': 512}
"""

"Combination 31/108: {'h1': 50, 'activations': 'tanh', 'lrs': 0.001, 'batch_sizes': 128}\nCombination 32/108: {'h1': 50, 'activations': 'tanh', 'lrs': 0.001, 'batch_sizes': 256}\nCombination 35/108: {'h1': 50, 'activations': 'tanh', 'lrs': 0.01, 'batch_sizes': 256}\nCombination 36/108: {'h1': 50, 'activations': 'tanh', 'lrs': 0.01, 'batch_sizes': 512}\nCombination 49/108: {'h1': 100, 'activations': 'tanh', 'lrs': 0.001, 'batch_sizes': 128}\nCombination 50/108: {'h1': 100, 'activations': 'tanh', 'lrs': 0.001, 'batch_sizes': 256}\nCombination 51/108: {'h1': 100, 'activations': 'tanh', 'lrs': 0.001, 'batch_sizes': 512}\nCombination 54/108: {'h1': 100, 'activations': 'tanh', 'lrs': 0.01, 'batch_sizes': 512}\nCombination 58/108: {'h1': 150, 'activations': 'relu', 'lrs': 0.001, 'batch_sizes': 128}\nCombination 59/108: {'h1': 150, 'activations': 'relu', 'lrs': 0.001, 'batch_sizes': 256}\nCombination 60/108: {'h1': 150, 'activations': 'relu', 'lrs': 0.001, 'batch_sizes': 512}\nCombination 67/1

In [17]:
#Best hyperparameters: {'h1': 150, 'activations': 'relu', 'lrs': 0.01, 'batch_sizes': 512}
#Final Test Accuracy after early stopping: 0.7144

In [18]:
#Best hyperparameters: {'h1': 10, 'activations': 'tanh', 'lrs': 0.01, 'batch_sizes': 128}
#Final Test Accuracy after early stopping: 0.7556